In [2]:
import torch
import json
import random
import datetime
import tqdm
from openai import OpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
HF_TOKEN = HF_TOKEN
OPENAI_API_KEY = openai_key

In [10]:
HF_MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # Options: "tiiuae/falcon-7b", "meta-llama/Llama-2-7b-chat-hf"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [3]:
# Define topics
TOPICS = [
    "Meeting Invitations", "Project Updates", "Financial Reports", "Customer Support",
    "Marketing Campaigns", "HR Announcements", "Sales Pitches", "Technical Documentation",
    "Event Invitations", "Security Alerts"
]

# Define demographic metadata
GENDERS = ["Male", "Female", "Non-binary"]
AGE_GROUPS = ["18-24", "25-34", "35-44", "45-54", "55+"]
ETHNICITIES = ["Asian", "Black", "Hispanic", "White", "Mixed", "Other"]
INDUSTRIES = ["Tech", "Finance", "Healthcare", "Education", "Retail", "Government"]
REGIONS = ["North America", "Europe", "Asia", "South America", "Africa", "Australia"]

# Toxic keywords (5% of emails will contain these)
TOXIC_KEYWORDS = ["scam", "idiot", "fraud", "stupid", "worthless"]

In [4]:
def clean_text(text):
    return text.strip()

In [5]:
# Function to generate email using HF model
def generate_email_hf(topic):
    prompt = f"""
    Generate a professional email about {topic}. The email should have:
    - A subject line starting with 'Subject:'
    - A professional greeting
    - A clear and concise body with 1-2 paragraphs
    - A professional closing with a name

    Output only the email content without explanations.
    """

    try:
        response = generator(prompt, max_length=512, temperature=0)
        return clean_text(response[0]["generated_text"])
    except Exception as e:
        print(f"Error generating email with Hugging Face model: {e}")
        return None

In [14]:
# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

def generate_email_openai(topic):
    prompt = f"""
    Generate a professional email about {topic}. The email should have:
    - A subject line starting with 'Subject:'
    - A professional greeting
    - A clear and concise body with 2-3 paragraphs
    - A professional closing with a name

    Output only the email content without explanations.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Use "gpt-4" or "gpt-4-turbo" depending on your access
            messages=[
                {"role": "system", "content": "You are an email writing assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return clean_text(response.choices[0].message.content)
    except Exception as e:
        print(f"Error generating email with OpenAI GPT-4: {e}")
        return None

In [15]:
emails = []
num_emails = 1

In [16]:
for i in tqdm.tqdm(range(num_emails)):
    topic = random.choice(TOPICS)
    demographics = {
        "gender": random.choice(GENDERS),
        "age_group": random.choice(AGE_GROUPS),
        "ethnicity": random.choice(ETHNICITIES),
        "industry": random.choice(INDUSTRIES),
        "region": random.choice(REGIONS)
    }

    # email_content = generate_email_hf(topic) if generator else None #-- Hugging Face model
    email_content = generate_email_openai(topic)

    subject = f"Important: {topic}"
    if "Subject:" in email_content:
        subject_line = email_content.split("Subject:")[1].split("\n")[0].strip()
        subject = subject_line if subject_line else subject

    is_toxic = random.random() < 0.05
    if is_toxic:
        toxic_word = random.choice(TOXIC_KEYWORDS)
        toxic_phrases = [
            f"This whole situation is {toxic_word}.",
            f"I think your approach is {toxic_word}.",
            f"The way this was handled is completely {toxic_word}.",
        ]
        email_content = email_content.replace("Regards,", f"{random.choice(toxic_phrases)}\n\nRegards,")

    email_date = (datetime.datetime.utcnow() -
                  datetime.timedelta(days=random.randint(0, 365),
                                     hours=random.randint(0, 23),
                                     minutes=random.randint(0, 59))).isoformat() + "Z"

    email_content = email_content.replace('\n', '<br>')

    email_json = {
        "metadata": {
            "message_id": f"email_{i+1:03d}",
            "from_email": f"sender{i}@example.com",
            "to": [f"recipient{i}@company.com"],
            "cc": [f"manager{i}@company.com"] if random.random() < 0.5 else [],
            "bcc": [f"hidden{i}@company.com"] if random.random() < 0.2 else [],
            "subject": subject,
            "date": email_date,
        },
        "content": {
            "html": f"<html><body><p>{email_content}</p></body></html>",
            "plain_text": email_content,
            "reference_summary": email_content[:200] + ("..." if len(email_content) > 200 else ""),
        },
        "analytics": {
            "demographics": demographics,
            "topic": topic,
            "is_toxic": is_toxic,
            "length": len(email_content),
        }
    }

    emails.append(email_json)

100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


In [17]:
emails

[{'metadata': {'message_id': 'email_001',
   'from_email': 'sender0@example.com',
   'to': ['recipient0@company.com'],
   'cc': [],
   'bcc': [],
   'subject': 'Enhancing Your Sales Pitches for Better Results',
   'date': '2024-03-24T13:36:14.955594Z'},
  'content': {'html': "<html><body><p>Subject: Enhancing Your Sales Pitches for Better Results<br><br>Dear [Recipient's Name],<br><br>I hope this email finds you well. As we continue to strive for excellence in our sales efforts, I wanted to touch base on the importance of refining our sales pitches. A well-crafted sales pitch is crucial in capturing the attention of potential clients and converting leads into customers. By focusing on the value proposition, addressing pain points, and showcasing how our offerings can solve specific challenges, we can make our sales pitches more effective and engaging.<br><br>To further enhance our sales pitches, let's consider personalizing the content to each prospect, utilizing compelling storytellin